# Version 21 — Sliding-mode / nonlinear HVAC control

**Experiment:** bounded sliding-mode feedback with physics-aware equipment dispatch.

Each zone combines temperature error with its filtered five-minute temperature change:

- Cooling surface: `s_c = T - upper_setpoint + 2 * rate`.
- Heating surface: `s_h = lower_setpoint - T - 2 * rate`.
- Nonlinear demand: `clip(surface / 0.8 K, 0, 1)`.

Unlike discontinuous ideal sliding control, the 0.8 K boundary layer reduces chattering. Each zone selects exactly one heating or cooling mode. Cooling damper demand is scaled by delivered-air temperature headroom; heating requests reheat and airflow. The AHU economizer, fan and hot-water coordination retain Version 7's thermodynamic dispatch, with corrected outdoor-temperature denormalization.

**Implementation constraints:** metadata-indexed observations/actions; float32 continuous and int32 discrete actions; O(AHUs + zones) per step; only NumPy and official Kaggle gateway imports. `reset()` clears per-zone history. Invalid temperature observations invalidate corresponding history.

**Limitations:** assumes regular five-minute sampling; equipment switching is not dwell-time constrained. The controller is unscored and needs official simulator validation for comfort tails, energy, carbon, and switching. This is a sliding-mode-inspired heuristic, not a proven robust sliding-mode controller with identified disturbance bounds.


In [ ]:
"""Version 21: boundary-layer sliding-mode HVAC control.

Zone feedback uses comfort-band error plus filtered temperature change:
    cooling_surface = temperature - upper_setpoint + horizon * rate
    heating_surface = lower_setpoint - temperature - horizon * rate

The continuous saturation sat(surface / boundary) limits switching noise.
A thermodynamic AHU model handles supply-air, mixing, and shared hot water.
Runtime: O(number of AHUs + number of zones) per five-minute decision.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Sliding-mode zone feedback with physics-aware shared HVAC dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    MIN_VAV_FRACTION = 0.10
    MIN_SLIDING_DEMAND = 0.05
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15
    # A zone starts conditioning once saturation exceeds 5% of full demand.
    # With an 0.8 K boundary this corresponds to a 0.04 K surface.
    BOUNDARY_K = 0.8
    RATE_HORIZON_STEPS = 2.0
    RATE_FILTER_ALPHA = 0.55
    MAX_VALID_TEMPERATURE_CHANGE_K = 1.5
    COOLING_HEADROOM_GAIN_K = 8.0
    COOLING_DAMPER_GAIN = 0.72
    HEATING_DAMPER_GAIN = 0.50
    HEATING_VALVE_GAIN = 0.74

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        outdoor_stats = meta.get(
            "observation_normalization_metadata", {}
        ).get("outside_air_temperature_sensor", {})
        self.outdoor_mean = float(
            outdoor_stats.get("sample_mean", float("nan"))
        )
        self.outdoor_std = float(
            outdoor_stats.get("sample_std", float("nan"))
        )

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Reset all zone temperature/rate history between buildings."""
        self._history = {}

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _outside_temperature(self, obs, index):
        """Return outdoor temperature in Kelvin, or NaN if unavailable."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")

        has_normalization = (
            np.isfinite(self.outdoor_mean)
            and np.isfinite(self.outdoor_std)
            and self.outdoor_std > 0
        )
        kelvin = (
            raw * self.outdoor_std + self.outdoor_mean
            if has_normalization else raw
        )
        return float(kelvin) if 210.0 <= kelvin <= 330.0 else float("nan")

    def _sliding_status(self, status):
        """Estimate temperature motion and attach cooling/heating demands.

        Rate is K per decision step (normally five minutes). An implausibly
        large jump is treated as a sensor discontinuity, resetting the rate
        rather than extrapolating from a faulty observation. This requires
        consecutive, evenly spaced calls by the official inference gateway.
        """
        zone_key = status["actions"]["temperature"]
        temperature = status["temp"]
        previous = self._history.get(zone_key)
        filtered_rate = 0.0

        if previous is not None:
            previous_temperature, previous_rate = previous
            temperature_change = temperature - previous_temperature
            if abs(temperature_change) <= self.MAX_VALID_TEMPERATURE_CHANGE_K:
                alpha = self.RATE_FILTER_ALPHA
                filtered_rate = (
                    alpha * temperature_change
                    + (1.0 - alpha) * previous_rate
                )

        self._history[zone_key] = (temperature, filtered_rate)

        cooling_surface = (
            temperature - status["high"]
            + self.RATE_HORIZON_STEPS * filtered_rate
        )
        heating_surface = (
            status["low"] - temperature
            - self.RATE_HORIZON_STEPS * filtered_rate
        )

        cooling_demand = self._clamp(
            cooling_surface / self.BOUNDARY_K, 0.0, 1.0
        )
        heating_demand = self._clamp(
            heating_surface / self.BOUNDARY_K, 0.0, 1.0
        )

        # Never heat and cool the same zone during the same decision.
        if cooling_demand >= heating_demand:
            heating_demand = 0.0
        else:
            cooling_demand = 0.0

        status["smc_cooling"] = cooling_demand
        status["smc_heating"] = heating_demand
        return status

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "low": low,
            "high": high,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["smc_cooling"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.MIN_SLIDING_DEMAND:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["smc_cooling"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.MIN_SLIDING_DEMAND
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(
        self, statuses, delivered_air_k, hot_water_demands, continuous
    ):
        """Convert sliding demands into bounded VAV damper/reheat commands.

        Cooling: scale opening by 8 K / available cooling headroom.
        Heating: request moderate airflow and use the reheat valve.
        A zero sliding demand returns the damper to its minimum opening.
        """
        total_opening = 0.0
        peak_opening = 0.0

        for status in statuses:
            zone = status["actions"]
            cooling = status["smc_cooling"]
            heating = status["smc_heating"]

            if cooling > 0.0:
                headroom_k = max(2.5, status["temp"] - delivered_air_k)
                airflow_gain = self._clamp(
                    self.COOLING_HEADROOM_GAIN_K / headroom_k, 0.45, 1.6
                )
                damper = self._clamp(
                    0.12 + self.COOLING_DAMPER_GAIN * cooling * airflow_gain,
                    self.MIN_VAV_FRACTION, 0.98
                )
            elif heating > 0.0:
                damper = self._clamp(
                    0.22 + self.HEATING_DAMPER_GAIN * heating, 0.18, 0.88
                )
                if zone["valve"] is not None:
                    valve = self._clamp(
                        0.12 + self.HEATING_VALVE_GAIN * heating, 0.0, 1.0
                    )
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append(
                        (max(status["cold"], 0.1 * heating), valve)
                    )
            else:
                damper = self.MIN_VAV_FRACTION

            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
            total_opening += damper
            peak_opening = max(peak_opening, damper)

        return total_opening / len(statuses), peak_opening

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_demand = max(max(z["smc_cooling"], z["smc_heating"]) for z in statuses)
        if worst_demand < self.MIN_SLIDING_DEMAND:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Run one controller step: zones -> sliding feedback -> AHU -> water.

        State contains only per-zone temperature and filtered rate.
        Equipment commands are recomputed every step; no online fitting.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = []
            for zone in ahu["zones"]:
                status = self._read_zone(obs, zone, fallback)
                if status is None:
                    # Do not use an outdated derivative after sensor recovery.
                    self._history.pop(zone["temperature"], None)
                    continue
                statuses.append(self._sliding_status(status))

            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor):
                outdoors.append(outdoor)
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}



In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
